In [ ]:
import csv
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image, ImageOps

REPO_DIR = Path("/content/fruit-classification-cnn_yolov5")
DATA_DIR = REPO_DIR / "fruit_classfication" / "data"
ZIP_PATH = DATA_DIR / "Fruit.zip"
RAW_DIR = Path("/content/fruit_raw_extracted")
TRAINVAL_DIR = Path("/content/fruit_trainval")
YOLOV5_DIR = Path("/content/yolov5")
MODELS_DIR = REPO_DIR / "fruit_classfication" / "models"
REPORTS_DIR = REPO_DIR / "fruit_classfication" / "reports"

BRANCH = "feat/build-YOLOv5-cls"
YOLOV5_COMMIT = "33ae15ef99bd04ce1d3f12155b8a85f355d73351"
SEED = 42
RATIO = {"train": 0.70, "val": 0.15, "test": 0.15}
CLASSES = ["Buoi", "Hong", "dudu"]
IMG_SIZE = 224
BATCH_SIZE = 32
WORKERS = 2
DEVICE = "0"

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/vantuann205/fruit-classification-cnn_yolov5.git", str(REPO_DIR)], check=True)
%cd {REPO_DIR}
!git branch --show-current
!nvidia-smi

import torch
assert torch.cuda.is_available()

In [ ]:
import zipfile

shutil.rmtree(RAW_DIR, ignore_errors=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as zf:
    for m in zf.infolist():
        m.filename = m.filename.replace("\\", "/")
        zf.extract(m, RAW_DIR)

for c in CLASSES:
    print(c, (RAW_DIR / "Fruit" / c).is_dir())

In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png"}
records = []
seen_hashes = {}
conflicts = []

for p in sorted(RAW_DIR.rglob("*")):
    if p.is_file() and p.suffix.lower() in VALID_EXTS:
        rel = p.relative_to(RAW_DIR / "Fruit")
        cls_name = rel.parts[0]
        if cls_name not in CLASSES:
            continue
        group = rel.parts[1] if len(rel.parts) > 2 else "root"
        with Image.open(p) as img:
            img.verify()
        with Image.open(p) as img:
            w, h = img.size
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        if digest in seen_hashes:
            if seen_hashes[digest] != cls_name:
                conflicts.append((digest, cls_name, seen_hashes[digest], str(p)))
            continue
        seen_hashes[digest] = cls_name
        records.append({
            "class": cls_name,
            "source": str(p.relative_to(RAW_DIR)),
            "source_group": group,
            "sha256": digest,
            "width": w,
            "height": h,
            "path": p
        })

assert not conflicts, f"Cross-class duplicate conflicts: {conflicts}"
print({c: sum(r["class"] == c for r in records) for c in CLASSES})

In [ ]:
for s in RATIO:
    shutil.rmtree(DATA_DIR / s, ignore_errors=True)
    for c in CLASSES:
        (DATA_DIR / s / c).mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)
manifest = []

for c in CLASSES:
    items = [r for r in records if r["class"] == c]
    rng.shuffle(items)
    n = len(items)
    n_tr = int(n * RATIO["train"])
    n_va = int(n * RATIO["val"])
    splits = {
        "train": items[:n_tr],
        "val": items[n_tr:n_tr + n_va],
        "test": items[n_tr + n_va:]
    }
    for s, subset in splits.items():
        for r in subset:
            with Image.open(r["path"]) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                w, h = im.size
                scale = 256 / max(w, h)
                im = im.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.Resampling.LANCZOS)
                out_path = DATA_DIR / s / c / f"{r['sha256'][:20]}.jpg"
                im.save(out_path, "JPEG", quality=92, optimize=True)
            manifest.append({
                "split": s,
                "class": c,
                "source": r["source"],
                "output": str(out_path.relative_to(DATA_DIR)),
                "source_sha256": r["sha256"],
                "source_group": r["source_group"],
                "width": r["width"],
                "height": r["height"]
            })

with (DATA_DIR / "split_manifest.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["split", "class", "source", "output", "source_sha256", "source_group", "width", "height"])
    writer.writeheader()
    writer.writerows(manifest)

CLASS_TO_IDX = {c: i for i, c in enumerate(sorted(CLASSES))}
(DATA_DIR / "class_to_idx.json").write_text(json.dumps(CLASS_TO_IDX, indent=2), encoding="utf-8")

In [ ]:
for s in RATIO:
    print(s, {c: len(list((DATA_DIR / s / c).glob("*.jpg"))) for c in CLASSES})

split_hashes = defaultdict(set)
for m in manifest:
    split_hashes[m["split"]].add(m["source_sha256"])

assert not (split_hashes["train"] & split_hashes["val"])
assert not (split_hashes["train"] & split_hashes["test"])
assert not (split_hashes["val"] & split_hashes["test"])

shutil.rmtree(TRAINVAL_DIR, ignore_errors=True)
shutil.copytree(DATA_DIR / "train", TRAINVAL_DIR / "train")
shutil.copytree(DATA_DIR / "val", TRAINVAL_DIR / "val")
assert (TRAINVAL_DIR / "train").is_dir()
assert (TRAINVAL_DIR / "val").is_dir()
assert not (TRAINVAL_DIR / "test").exists()
print("Staging verified:", TRAINVAL_DIR)

In [ ]:
if not YOLOV5_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLOV5_DIR)], check=True)
subprocess.run(["git", "-C", str(YOLOV5_DIR), "checkout", YOLOV5_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(YOLOV5_DIR / "requirements.txt")], check=True)

import torch
assert torch.cuda.is_available()

In [ ]:
SMOKE_PROJECT = YOLOV5_DIR / "runs" / "train-cls"
SMOKE_NAME = "smoke_1ep"
SMOKE_DIR = SMOKE_PROJECT / SMOKE_NAME
shutil.rmtree(SMOKE_DIR, ignore_errors=True)

subprocess.run([
    sys.executable, str(YOLOV5_DIR / "classify" / "train.py"),
    "--model", "yolov5s-cls.pt",
    "--data", str(TRAINVAL_DIR),
    "--epochs", "1",
    "--img", str(IMG_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--workers", str(WORKERS),
    "--seed", str(SEED),
    "--project", str(SMOKE_PROJECT),
    "--name", SMOKE_NAME,
    "--exist-ok"
], cwd=YOLOV5_DIR, check=True)

assert (SMOKE_DIR / "weights" / "best.pt").exists()
print("Smoke test passed:", SMOKE_DIR)

In [ ]:
def run_experiment(name, lr, epochs=30):
    proj = YOLOV5_DIR / "runs" / "train-cls"
    exp_dir = proj / name
    shutil.rmtree(exp_dir, ignore_errors=True)
    start_utc = datetime.now(timezone.utc)
    t0 = time.perf_counter()
    subprocess.run([
        sys.executable, str(YOLOV5_DIR / "classify" / "train.py"),
        "--model", "yolov5s-cls.pt",
        "--data", str(TRAINVAL_DIR),
        "--epochs", str(epochs),
        "--img", str(IMG_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--optimizer", "Adam",
        "--lr0", str(lr),
        "--device", DEVICE,
        "--workers", str(WORKERS),
        "--seed", str(SEED),
        "--project", str(proj),
        "--name", name,
        "--exist-ok"
    ], cwd=YOLOV5_DIR, check=True)
    t1 = time.perf_counter()
    end_utc = datetime.now(timezone.utc)
    sec = t1 - t0
    best_pt = exp_dir / "weights" / "best.pt"
    assert best_pt.exists()

    val_acc = 0.0
    results_path = exp_dir / "results.csv"
    if results_path.exists():
        with results_path.open("r", encoding="utf-8") as f:
            reader = csv.reader(f)
            header = [h.strip() for h in next(reader)]
            top1_idx = -1
            for col in ["metrics/accuracy_top1", "top1_acc"]:
                if col in header:
                    top1_idx = header.index(col)
                    break
            if top1_idx != -1:
                vals = [float(row[top1_idx].strip()) for row in reader if len(row) > top1_idx]
                val_acc = max(vals) if vals else 0.0

    return {
        "name": name,
        "lr": lr,
        "epochs": epochs,
        "dir": exp_dir,
        "best_pt": best_pt,
        "start_utc": start_utc.isoformat(),
        "end_utc": end_utc.isoformat(),
        "seconds": sec,
        "val_acc": val_acc
    }

RUN_BASELINE = True
if RUN_BASELINE:
    exp01_res = run_experiment("baseline_lr001", lr=0.001, epochs=30)
    print("EXP-01 finished:", exp01_res["val_acc"])

In [ ]:
RUN_EXP02 = True
if RUN_EXP02:
    exp02_res = run_experiment("exp02_lr0005", lr=0.0005, epochs=30)
    print("EXP-02 finished:", exp02_res["val_acc"])

In [ ]:
experiments = [e for e in [exp01_res, exp02_res] if e is not None]
best_exp = max(experiments, key=lambda x: x["val_acc"])
print(f"Best: {best_exp['name']} (Val Top-1: {best_exp['val_acc']:.4f})")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = MODELS_DIR / "yolov5_cls_best.pt"
shutil.copy2(best_exp["best_pt"], best_model_path)
assert best_model_path.exists()
print("Saved best checkpoint:", best_model_path, f"{best_model_path.stat().st_size} bytes")

In [ ]:
val_proj = YOLOV5_DIR / "runs" / "val-cls"
val_name = "final_test"
val_dir = val_proj / val_name
shutil.rmtree(val_dir, ignore_errors=True)

eval_proc = subprocess.run([
    sys.executable, str(YOLOV5_DIR / "classify" / "val.py"),
    "--data", str(DATA_DIR),
    "--weights", str(best_model_path),
    "--img", str(IMG_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--project", str(val_proj),
    "--name", val_name,
    "--exist-ok",
    "--verbose"
], cwd=YOLOV5_DIR, capture_output=True, text=True, check=True)

print(eval_proc.stdout)

In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = REPORTS_DIR / "yolov5_runs.md"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unknown"

report_lines = [
    "# YOLOv5-cls Experiment Report\n",
    "## 1. Dataset",
    f"- Source: Fruit.zip (classes: {', '.join(CLASSES)})",
    f"- Input size: {IMG_SIZE}x{IMG_SIZE}\n",
    "## 2. Dataset Split",
    f"- Train: {RATIO['train']*100:.0f}%",
    f"- Val: {RATIO['val']*100:.0f}%",
    f"- Test: {RATIO['test']*100:.0f}%",
    f"- Random Seed: {SEED}",
    "- Manifest: split_manifest.csv\n",
    "## 3. Class Mapping",
    f"- Mapping: {json.dumps(CLASS_TO_IDX)}\n",
    "## 4. Model Architecture",
    "- Backbone: YOLOv5s-cls (yolov5s-cls.pt)",
    f"- Commit: {YOLOV5_COMMIT}\n",
    "## 5. Environment",
    f"- GPU: {gpu_name}",
    f"- Device: {DEVICE}",
    f"- Workers: {WORKERS}\n",
    "## 6. Training Configuration",
    f"- Batch size: {BATCH_SIZE}",
    f"- Image size: {IMG_SIZE}",
    "- Optimizer: Adam\n",
    "## 7. Experiments",
    "| Experiment | Epochs | Learning Rate | Batch Size | Wall Time (s) | Val Top-1 Acc |",
    "|---|---|---|---|---|---|"
]

for exp in experiments:
    report_lines.append(f"| {exp['name']} | {exp['epochs']} | {exp['lr']} | {BATCH_SIZE} | {exp['seconds']:.2f} | {exp['val_acc']:.4f} |")

report_lines.extend([
    "\n## 8. Training Time",
    *[f"- **{exp['name']}**: {exp['seconds']:.2f}s ({exp['start_utc']} to {exp['end_utc']})" for exp in experiments],
    "\n## 9. Validation Results",
    f"- Best Experiment: {best_exp['name']}",
    f"- Best Val Top-1 Accuracy: {best_exp['val_acc']:.4f}\n",
    "## 10. Test Results",
    f"- Checkpoint: {best_model_path.name}",
    "- Evaluation Log:\n```",
    eval_proc.stdout.strip(),
    "```\n",
    "## 11. Selected Checkpoint",
    "- Path: `fruit_classfication/models/yolov5_cls_best.pt`",
    f"- Size: {best_model_path.stat().st_size} bytes",
    f"- Selected from: {best_exp['name']}\n",
    "## 12. Reproducibility",
    f"- Seed: {SEED}",
    "- SHA256 manifest: split_manifest.csv",
    "- Staged train/val without test leakage"
])

report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Report generated:", report_path)